<a href="https://colab.research.google.com/github/Huii0529/Data-Science-Project/blob/main/Week14_DataAggregation_20260620.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Week 14: Data Aggregation and Group Operations**

This notebook introduces how to summarise and analyse data by groups using pandas.

Data aggregation is useful when we want to answer questions such as:

- What is the average sales by region?
- What is the maximum tip by day?
- How many observations belong to each category?
- How do different groups compare with each other?

The main idea is based on the **split-apply-combine** strategy:

1. **Split** the data into groups.
2. **Apply** a function to each group.
3. **Combine** the results into a summary table.

Reference:  
Wes McKinney, ***Python for Data Analysis***, Chapter 10.


## **Learning Outcomes**
By the end of this notebook, students should be able to:

1. Explain the split-apply-combine strategy in pandas.
2. Use **`groupby()`** to summarise data by one or more categorical variables.
3. Apply single and multiple aggregation functions.
4. Understand the difference between **`agg()`**, **`apply()`**, and **`transform()`**.
5. Create pivot tables and cross-tabulations for summarising grouped data.

### **Import Required Libraries**

We use:

- **`NumPy`** for numerical operations and random number generation.
- **`pandas`** for creating DataFrames and performing group-based analysis.

In [1]:
import numpy as np
import pandas as pd

## **14.1 Group Operations**
The most important idea in group operations is **split-apply-combine**.

| Step | Meaning | Example |
|---|---|---|
| Split | Divide the dataset into groups | Group customers by day |
| Apply | Apply a function to each group | Calculate mean, max, sum |
| Combine | Combine the results into one output | Produce a summary table |

In pandas, this is usually done using:

```python
df.groupby(...).agg(...)

In [2]:
# Create an example DataFrame
# key1 and key2 are grouping variables
# data1 and data2 are numerical variables to be summarised

rng = np.random.default_rng(seed=12345)

df = pd.DataFrame(
    {"key1" : ["a", "a", None, "b", "b", "a", None],
     "key2" : pd.Series([1, 2, 1, 2, 1, None, 1], dtype="Int64"),
     "data1" : rng.standard_normal(7),
     "data2" : rng.standard_normal(7)
})

df

,key1,key2,data1,data2
0,a,1,-1.423825,0.648893
1,a,2,1.263728,0.361058
2,None,1,-0.870662,-1.952863
3,b,2,-0.259173,2.347410
4,b,1,-0.075343,0.968497
5,a,<NA>,-0.740885,-0.759387
6,None,1,-1.367793,0.902198


In [3]:
# Group the values in data1 according to the categories in key1
# This creates a GroupBy object, but does not calculate anything yet
grouped = df["data1"].groupby(df["key1"])

#### **Key Take Away:**
1. The **`groupby()`** operation only prepares the grouping structure.  
2. No summary is calculated until we apply a function such as **`mean()`**, **`max()`**, or **`sum()`**.

In [4]:
# Calculate the maximum value of data1 for each group in key1
grouped.max()

,data1
key1,
a,1.263728
b,-0.075343


In [5]:
# Same result as the previous two cells, but written in one line
df["data1"].groupby(df["key1"]).max()

,data1
key1,
a,1.263728
b,-0.075343


#### **Key Take Away**
The previous example shows that pandas allows both styles:

1. Create a GroupBy object first, then apply a function.
2. Perform grouping and aggregation directly in one line.

In [6]:
# Group data1 by two keys: key1 and key2
# Then calculate the mean for each combination of key1 and key2
means = df["data1"].groupby([df["key1"], df["key2"]]).mean()
means

key1  key2
a     1      -1.423825
      2       1.263728
b     1      -0.075343
      2      -0.259173
Name: data1, dtype: float64

In [7]:
# Convert the inner row index key2 into columns
# This makes the result easier to read as a table
# Change the view of table from long format to wide format
means.unstack()

key2,1,2
key1,,
a,-1.423825,1.263728
b,-0.075343,-0.259173


#### **Key Take Away**
1. **`unstack()`** is useful when the grouped result has a MultiIndex.  
2. It converts one level of the row index into columns, producing a wider table.

In [8]:
# Group the DataFrame by key1
# pandas calculates the mean only for numerical columns
df.groupby("key1").mean()

,key2,data1,data2
key1,,,
a,1.5,-0.300327,0.083521
b,1.5,-0.167258,1.657953


In [9]:
# Group the DataFrame by two columns: key1 and key2
# The result has a MultiIndex because there are two grouping variables
df.groupby(["key1", "key2"]).mean()

data1     data2
key1 key2                    
a    1    -1.423825  0.648893
     2     1.263728  0.361058
b    1    -0.075343  0.968497
     2    -0.259173  2.347410

#### **Difference Between `size()` and `count()`**

1. Both **`size()`** and **`count()`** are used to count data, but they are not exactly the same.

| Method | What it counts |
|---|---|
| `size()` | Counts the number of rows in each group, including rows with missing values |
| `count()` | Counts non-missing values for each column |

In [10]:
# Display df content
df

,key1,key2,data1,data2
0,a,1,-1.423825,0.648893
1,a,2,1.263728,0.361058
2,None,1,-0.870662,-1.952863
3,b,2,-0.259173,2.347410
4,b,1,-0.075343,0.968497
5,a,<NA>,-0.740885,-0.759387
6,None,1,-1.367793,0.902198


In [11]:
# Count the number of rows in each group
df.groupby(["key1", "key2"]).size()

key1  key2
a     1       1
      2       1
b     1       1
      2       1
dtype: int64

### **Notes:**
- pandas exclude groups where the grouping key contains missing values
- By default, groupby() uses:

```python
dropna=True
```

In [12]:
# Count the number of non-missing values in data1 for each group
df.groupby(["key1", "key2"])["data1"].count()

key1  key2
a     1       1
      2       1
b     1       1
      2       1
Name: data1, dtype: int64

In [13]:
# By default, missing values in grouping keys are excluded
df.groupby("key1").size()

,0
key1,
a,3
b,2


In [14]:
# Use dropna=False to include missing values as their own group
df.groupby(["key1", "key2"], dropna=False).size()

key1  key2
a     1       1
      2       1
      <NA>    1
b     1       1
      2       1
NaN   1       2
dtype: int64

In [15]:
# Recreate the DataFrame using a fixed random seed
# This makes the output reproducible every time the notebook is run

rng = np.random.default_rng(seed=12345)

df = pd.DataFrame({
    "key1" : ["a", "a", None, "b", "b", "a", None],
    "key2" : pd.Series([1, 2, 1, 2, 1, None, 1], dtype="Int64"),
    "data1" : rng.standard_normal(7),
    "data2" : rng.standard_normal(7)
})

print("Original DataFrame:")
print(df)

print("\nsize(): counts the number of rows in each group")
print(df.groupby("key1").size())

print("\ncount(): counts non-missing values in each column")
print(df.groupby("key1").count())

Original DataFrame:
   key1  key2     data1     data2
0     a     1 -1.423825  0.648893
1     a     2  1.263728  0.361058
2  None     1 -0.870662 -1.952863
3     b     2 -0.259173  2.347410
4     b     1 -0.075343  0.968497
5     a  <NA> -0.740885 -0.759387
6  None     1 -1.367793  0.902198

size(): counts the number of rows in each group
key1
a    3
b    2
dtype: int64

count(): counts non-missing values in each column
      key2  data1  data2
key1                    
a        2      3      3
b        2      2      2


#### **Iterating Over Groups**

When we iterate over a **`GroupBy`** object, pandas returns two items:

1. The group name.
2. The subset of the DataFrame that belongs to that group.

In [16]:
# Iterate over each group in key1
for group_name, group_df in df.groupby("key1"):
    print("Group name:", group_name)
    print(group_df)
    print()

Group name: a
  key1  key2     data1     data2
0    a     1 -1.423825  0.648893
1    a     2  1.263728  0.361058
5    a  <NA> -0.740885 -0.759387

Group name: b
  key1  key2     data1     data2
3    b     2 -0.259173  2.347410
4    b     1 -0.075343  0.968497



In [17]:
# Convert each group into a dictionary entry
# The dictionary key is the group name
# The dictionary value is the DataFrame for that group
pieces = {name: group for name, group in df.groupby("key1")}

#### **Key Take Away**
1. This allows us to access each group directly using dictionary-style indexing.

In [18]:
# Accessing item in dictionary
pieces["a"]

,key1,key2,data1,data2
0,a,1,-1.423825,0.648893
1,a,2,1.263728,0.361058
5,a,<NA>,-0.740885,-0.759387


#### **Selecting One or More Columns After Grouping**

After grouping, we can select:

1. One column using single square brackets: **returns a Series**.
2. One or more columns using double square brackets: **returns a DataFrame**.

In [19]:
# Double square brackets return a DataFrame
df.groupby(["key1", "key2"])[["data2"]].mean()

data2
key1 key2          
a    1     0.648893
     2     0.361058
b    1     0.968497
     2     2.347410

In [20]:
# Single square brackets return a Series
df.groupby(["key1", "key2"])["data2"].mean()

key1  key2
a     1       0.648893
      2       0.361058
b     1       0.968497
      2       2.347410
Name: data2, dtype: float64

### **Grouping with Dictionaries and Series**

In [21]:
# Example DataFrame
# Setting the seed number to ensure reproducibility
rng = np.random.default_rng(seed=12345)
people = pd.DataFrame(rng.standard_normal((5, 5)),
                      columns=["a", "b", "c", "d", "e"],
                      index=["Joe", "Steve", "Wanda", "Jill", "Trey"])
people

,a,b,c,d,e
Joe,-1.423825,1.263728,-0.870662,-0.259173,-0.075343
Steve,-0.740885,-1.367793,0.648893,0.361058,-1.952863
Wanda,2.347410,0.968497,-0.759387,0.902198,-0.466953
Jill,-0.060690,0.788844,-1.256668,0.575858,1.398979
Trey,1.322298,-0.299699,0.902919,-1.621583,-0.158189


In [22]:
# Add a few NA values using loc function
people.loc["Wanda", ["b", "c"]] = np.nan
people

,a,b,c,d,e
Joe,-1.423825,1.263728,-0.870662,-0.259173,-0.075343
Steve,-0.740885,-1.367793,0.648893,0.361058,-1.952863
Wanda,2.347410,NaN,NaN,0.902198,-0.466953
Jill,-0.060690,0.788844,-1.256668,0.575858,1.398979
Trey,1.322298,-0.299699,0.902919,-1.621583,-0.158189


In [23]:
# Add missing values using integer positions
# Rows at positions 4 and 1, columns at positions 0 and 3, are set to NaN
people.iloc[[4, 1], [0, 3]] = np.nan

people

,a,b,c,d,e
Joe,-1.423825,1.263728,-0.870662,-0.259173,-0.075343
Steve,NaN,-1.367793,0.648893,NaN,-1.952863
Wanda,2.347410,NaN,NaN,0.902198,-0.466953
Jill,-0.060690,0.788844,-1.256668,0.575858,1.398979
Trey,NaN,-0.299699,0.902919,NaN,-0.158189


### **Grouping Columns Using a Dictionary**

A dictionary can be used to assign columns into larger groups.

For example:

- columns **`a`**, **`b`**, and **`e`** belong to the **`"red"`** group
- columns **`c`** and **`d`** belong to the **`"blue"`** group

Since **`groupby()`** usually groups rows, we first transpose the DataFrame using **`.T`**.  
This temporarily converts columns into rows so that the mapping can be applied.

In [26]:
# Define a mapping that assigns each existing column to a group
# Mapping was refering to the row indexing

mapping = {
    "a": "red",
    "b": "red",
    "c": "blue",
    "d": "blue",
    "e": "red"
}

In [28]:
# Transpose the DataFrame so that columns become rows
# Then group the columns according to the mapping dictionary
people.T.groupby(mapping).sum().T

,blue,red
Joe,-1.129835,-0.235440
Steve,0.648893,-3.320656
Wanda,0.902198,1.880456
Jill,-0.680811,2.127134
Trey,0.902919,-0.457888


In [29]:
# Add two new columns, f and g

rng = np.random.default_rng(seed=12345)

people = people.assign(
    f = rng.standard_normal(5),
    g = rng.standard_normal(5)
)

people

,a,b,c,d,e,f,g
Joe,-1.423825,1.263728,-0.870662,-0.259173,-0.075343,-1.423825,-0.740885
Steve,NaN,-1.367793,0.648893,NaN,-1.952863,1.263728,-1.367793
Wanda,2.347410,NaN,NaN,0.902198,-0.466953,-0.870662,0.648893
Jill,-0.060690,0.788844,-1.256668,0.575858,1.398979,-0.259173,0.361058
Trey,NaN,-0.299699,0.902919,NaN,-0.158189,-0.075343,-1.952863


In [30]:
# Update the mapping after adding new columns f and g

mapping = {
    "a": "red",
    "b": "red",
    "c": "blue",
    "d": "blue",
    "e": "red",
    "f": "orange",
    "g": "violet"
}

people.T.groupby(mapping).sum().T

,blue,orange,red,violet
Joe,-1.129835,-1.423825,-0.235440,-0.740885
Steve,0.648893,1.263728,-3.320656,-1.367793
Wanda,0.902198,-0.870662,1.880456,0.648893
Jill,-0.680811,-0.259173,2.127134,0.361058
Trey,0.902919,-0.075343,-0.457888,-1.952863


### **Grouping with Functions**

In [31]:
# Display people content
people

,a,b,c,d,e,f,g
Joe,-1.423825,1.263728,-0.870662,-0.259173,-0.075343,-1.423825,-0.740885
Steve,NaN,-1.367793,0.648893,NaN,-1.952863,1.263728,-1.367793
Wanda,2.347410,NaN,NaN,0.902198,-0.466953,-0.870662,0.648893
Jill,-0.060690,0.788844,-1.256668,0.575858,1.398979,-0.259173,0.361058
Trey,NaN,-0.299699,0.902919,NaN,-0.158189,-0.075343,-1.952863


In [32]:
# Group rows based on the length of the row index name
# For example, "Joe" has length 3, while "Steve" has length 5
people.groupby(len).sum()

,a,b,c,d,e,f,g
3,-1.423825,1.263728,-0.870662,-0.259173,-0.075343,-1.423825,-0.740885
4,-0.060690,0.489146,-0.353749,0.575858,1.240790,-0.334517,-1.591805
5,2.347410,-1.367793,0.648893,0.902198,-2.419816,0.393067,-0.718900


#### **Key Take Away**
In this example, `len` is applied to each row index label.

For example:

- `"Joe"` has length 3
- `"Jill"` has length 4
- `"Steve"` has length 5

Therefore, pandas groups rows according to the number of characters in the index name.

## **14.2 Data Aggregation**
Aggregation means reducing many values into one summary value.

Common aggregation functions include:

| Function | Meaning |
|---|---|
| `mean()` | Average value |
| `sum()` | Total value |
| `min()` | Minimum value |
| `max()` | Maximum value |
| `count()` | Number of non-missing values |
| `std()` | Standard deviation |

In [33]:
# From previous example
df

,key1,key2,data1,data2
0,a,1,-1.423825,0.648893
1,a,2,1.263728,0.361058
2,None,1,-0.870662,-1.952863
3,b,2,-0.259173,2.347410
4,b,1,-0.075343,0.968497
5,a,<NA>,-0.740885,-0.759387
6,None,1,-1.367793,0.902198


In [34]:
# For each group in key1, return the three smallest values of data1
# This does not return one summary value per group
# Instead, it can return multiple rows per group
grouped = df.groupby("key1")
grouped["data1"].nsmallest(3)

key1   
a     0   -1.423825
      5   -0.740885
      1    1.263728
b     3   -0.259173
      4   -0.075343
Name: data1, dtype: float64

#### **Key Take Away**
Unlike `mean()`, `sum()`, or `max()`, the `nsmallest()` method does not reduce each group into one value.  
- Instead, it returns the smallest `n` values from each group.

In [36]:
# Define a custom aggregation function
# This function calculates the range: maximum value - minimum value
# arr - variable can be change accordingly

def range_value(arr):
    return arr.max() - arr.min()

In [37]:
# Apply the custom aggregation function to each group
grouped.agg(range_value)

,key2,data1,data2
key1,,,
a,1,2.687553,1.408280
b,1,0.183830,1.378913


In [38]:
# Generate descriptive statistics for each group
# This includes count, mean, standard deviation, min, quartiles, and max
grouped.describe()

key2                                           data1            ...  \
     count mean       std  min   25%  50%   75%  max count      mean  ...   
key1                                                                  ...   
a      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   3.0 -0.300327  ...   
b      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   2.0 -0.167258  ...   

                         data2                                          \
           75%       max count      mean       std       min       25%   
key1                                                                     
a     0.261422  1.263728   3.0  0.083521  0.744032 -0.759387 -0.199165   
b    -0.121301 -0.075343   2.0  1.657953  0.975039  0.968497  1.313225   

                                    
           50%       75%       max  
key1                                
a     0.361058  0.504975  0.648893  
b     1.657953  2.002681  2.347410  

[2 rows x 24 columns]

### **Column-Wise and Multiple Function Application**

In [39]:
# Load the tips dataset
# This dataset contains restaurant bill information such as:
# total_bill, tip, sex, smoker, day, time, and size
tips = pd.read_csv("https://bit.ly/3VyE0vP")
tips.sample(5)

,total_bill,tip,smoker,day,time,size
167,31.71,4.50,No,Sun,Dinner,4
206,26.59,3.41,Yes,Sat,Dinner,3
230,24.01,2.00,Yes,Sat,Dinner,4
66,16.45,2.47,No,Sat,Dinner,2
72,26.86,3.14,Yes,Sat,Dinner,2


In [40]:
# Calculate tip percentage
# This measures the tip as a proportion of the total bill
tips["tip_pct"] = tips["tip"] / tips["total_bill"]
tips.sample(5)

,total_bill,tip,smoker,day,time,size,tip_pct
157,25.00,3.75,No,Sun,Dinner,4,0.150000
0,16.99,1.01,No,Sun,Dinner,2,0.059447
2,21.01,3.50,No,Sun,Dinner,3,0.166587
49,18.04,3.00,No,Sun,Dinner,2,0.166297
46,22.23,5.00,No,Sun,Dinner,2,0.224921


In [41]:
# Group the tips dataset by day and smoker status
# Each group represents one combination, such as:
# Friday + smoker, Friday + non-smoker, Saturday + smoker, and so on

grouped = tips.groupby(["day", "smoker"])

In [44]:
# Select the tip_pct column from the grouped data
grouped_pct = grouped["tip_pct"]

# Calculate the mean tip percentage for each group
grouped_pct.agg("mean")

day   smoker
Fri   No        0.151650
      Yes       0.174783
Sat   No        0.158048
      Yes       0.147906
Sun   No        0.160113
      Yes       0.187250
Thur  No        0.160298
      Yes       0.163863
Name: tip_pct, dtype: float64

In [46]:
# Apply multiple aggregation functions to tip_pct
# Refer to tip_pct - tips percentage
grouped_pct.agg(["mean", "std", range_value])

mean       std  range_value
day  smoker                                 
Fri  No      0.151650  0.028123     0.067349
     Yes     0.174783  0.051293     0.159925
Sat  No      0.158048  0.039767     0.235193
     Yes     0.147906  0.061375     0.290095
Sun  No      0.160113  0.042347     0.193226
     Yes     0.187250  0.154134     0.644685
Thur No      0.160298  0.038774     0.193350
     Yes     0.163863  0.039389     0.151240

In [49]:
# Rename the output columns while applying aggregation functions
# (name, order) - parenthesis >> tuple (bag, sequential order)
grouped_pct.agg([
    ("mean_tip_pct", "mean"),
    ("std_tip_pct", "std")
])

mean_tip_pct  std_tip_pct
day  smoker                           
Fri  No          0.151650     0.028123
     Yes         0.174783     0.051293
Sat  No          0.158048     0.039767
     Yes         0.147906     0.061375
Sun  No          0.160113     0.042347
     Yes         0.187250     0.154134
Thur No          0.160298     0.038774
     Yes         0.163863     0.039389

In [51]:
# Apply the same list of functions to multiple columns
functions = ["count", "mean", "max", "min", "median"]

result = grouped[["tip_pct", "total_bill"]].agg(functions) # double bracket is for extracting the data and make it as a dataframe

result

tip_pct                                         total_bill  \
              count      mean       max       min    median      count   
day  smoker                                                              
Fri  No           4  0.151650  0.187735  0.120385  0.149241          4   
     Yes         15  0.174783  0.263480  0.103555  0.173913         15   
Sat  No          45  0.158048  0.291990  0.056797  0.150152         45   
     Yes         42  0.147906  0.325733  0.035638  0.153624         42   
Sun  No          57  0.160113  0.252672  0.059447  0.161665         57   
     Yes         19  0.187250  0.710345  0.065660  0.138122         19   
Thur No          45  0.160298  0.266312  0.072961  0.153492         45   
     Yes         17  0.163863  0.241255  0.090014  0.153846         17   

                                              
                  mean    max    min  median  
day  smoker                                   
Fri  No      18.420000  22.75  12.46  19.235  
     Yes     16.813333  40.17   5.75  13.420  
Sat  No      19.661778  48.33   7.25  17.820  
     Yes     21.276667  50.81   3.07  20.390  
Sun  No      20.506667  48.17   8.77  18.430  
     Yes     24.120000  45.35   7.25  23.100  
Thur No      17.113111  41.19   7.51  15.950  
     Yes     19.190588  43.11  10.34  16.470

#### **Key Takeaway**

The result has **hierarchical column names**, also known as **MultiIndex columns**.

- This means the output table has more than one level of column headings.

- For example:

```text
                tip_pct                      total_bill
                count   mean   max   min      count   mean   max   min
day smoker
Fri No           ...    ...    ...   ...       ...    ...    ...   ...
Fri Yes          ...    ...    ...   ...       ...    ...    ...   ...
```

In [54]:
# Passing a list of tuples to rename aggregation output columns
ftuples = [("Mean", "mean"), ("Variance", "var")] # (name of the function, function)
grouped[["tip_pct", "total_bill"]].agg(ftuples)

tip_pct           total_bill            
                 Mean  Variance       Mean    Variance
day  smoker                                           
Fri  No      0.151650  0.000791  18.420000   25.596333
     Yes     0.174783  0.002631  16.813333   82.562438
Sat  No      0.158048  0.001581  19.661778   79.908965
     Yes     0.147906  0.003767  21.276667  101.387535
Sun  No      0.160113  0.001793  20.506667   66.099980
     Yes     0.187250  0.023757  24.120000  109.046044
Thur No      0.160298  0.001503  17.113111   59.625081
     Yes     0.163863  0.001551  19.190588   69.808518

In [57]:
# Display original dataset
tips

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808
...,...,...,...,...,...,...,...
239,29.03,5.92,No,Sat,Dinner,3,0.203927
240,27.18,2.00,Yes,Sat,Dinner,2,0.073584
241,22.67,2.00,Yes,Sat,Dinner,2,0.088222
242,17.82,1.75,No,Sat,Dinner,2,0.098204


In [56]:
# Apply different aggregation functions to different columns
# and assign custom names to the output columns
grouped_res = grouped.agg(
    max_tip=("tip", "max"), # referring to tip from the dataset and get the max value
    total_gp_size=("size", "sum") # refrring to the total_gp_size and get the sum value
)

grouped_res

max_tip  total_gp_size
day  smoker                        
Fri  No         3.50              9
     Yes        4.73             31
Sat  No         9.00            115
     Yes       10.00            104
Sun  No         6.00            167
     Yes        6.50             49
Thur No         6.70            112
     Yes        5.00             40

#### **Key Take Away**
The format for named aggregation is:

```python
new_column_name = ("original_column_name", "aggregation_function")
```

In [58]:
# Apply multiple functions to total_bill
# Apply one function to size

grouped.agg({
    "total_bill" : ["min", "max", "mean", "std"],
    "size" : "sum"
})

total_bill                              size
                   min    max       mean        std  sum
day  smoker                                             
Fri  No          12.46  22.75  18.420000   5.059282    9
     Yes          5.75  40.17  16.813333   9.086388   31
Sat  No           7.25  48.33  19.661778   8.939181  115
     Yes          3.07  50.81  21.276667  10.069138  104
Sun  No           8.77  48.17  20.506667   8.130189  167
     Yes          7.25  45.35  24.120000  10.442511   49
Thur No           7.51  41.19  17.113111   7.721728  112
     Yes         10.34  43.11  19.190588   8.355149   40

### **Returning Aggregated Data Without Row Indexes**

In [59]:
# Return the grouped result as a regular DataFrame
# as_index=False keeps the group labels as normal columns
tips.groupby(["day", "smoker"], as_index=False)[
    ['total_bill', 'tip', 'size', 'tip_pct']
].mean()

,day,smoker,total_bill,tip,size,tip_pct
0,Fri,No,18.420000,2.812500,2.250000,0.151650
1,Fri,Yes,16.813333,2.714000,2.066667,0.174783
2,Sat,No,19.661778,3.102889,2.555556,0.158048
3,Sat,Yes,21.276667,2.875476,2.476190,0.147906
4,Sun,No,20.506667,3.167895,2.929825,0.160113
5,Sun,Yes,24.120000,3.516842,2.578947,0.187250
6,Thur,No,17.113111,2.673778,2.488889,0.160298
7,Thur,Yes,19.190588,3.030000,2.352941,0.163863


## **14.3 Apply: General split-apply-combine**
The **`apply()`** method is more flexible than **`agg()`**.

Use **`apply()`** when the operation:

- returns multiple rows per group,
- uses custom logic,
- cannot be easily expressed using standard aggregation functions.

In [60]:
# Define a function that returns the top n rows based on a selected column
def top(df, n=5, column="tip_pct"): # refering to df, only want 5 data, extract from tip_pct
    return df.sort_values(column, ascending=False).head(n) # Descending order

In [64]:
# Test the function on the full tips dataset
# Here, we return the top 6 rows based on total_bill

top(tips, n=6, column="total_bill")

,total_bill,tip,smoker,day,time,size,tip_pct
170,50.81,10.00,Yes,Sat,Dinner,3,0.196812
212,48.33,9.00,No,Sat,Dinner,4,0.186220
59,48.27,6.73,No,Sat,Dinner,4,0.139424
156,48.17,5.00,No,Sun,Dinner,6,0.103799
182,45.35,3.50,Yes,Sun,Dinner,3,0.077178
102,44.30,2.50,Yes,Sat,Dinner,3,0.056433


In [65]:
# Apply the top function separately to each smoker group
tips.groupby("smoker")[['total_bill', 'tip', 'size', 'tip_pct']].apply(top)

total_bill   tip  size   tip_pct
smoker                                      
No     232       11.61  3.39     2  0.291990
       149        7.51  2.00     2  0.266312
       51        10.29  2.60     2  0.252672
       185       20.69  5.00     5  0.241663
       88        24.71  5.85     2  0.236746
Yes    172        7.25  5.15     2  0.710345
       178        9.60  4.00     2  0.416667
       67         3.07  1.00     1  0.325733
       183       23.17  6.50     4  0.280535
       109       14.31  4.00     2  0.279525

In [66]:
# Return the top 3 rows for each smoker group based on total_bill
tips.groupby("smoker")[['total_bill', 'tip', 'size', 'tip_pct']].apply(
    top,
    n=3, column="total_bill"
)

total_bill    tip  size   tip_pct
smoker                                       
No     212       48.33   9.00     4  0.186220
       59        48.27   6.73     4  0.139424
       156       48.17   5.00     6  0.103799
Yes    170       50.81  10.00     3  0.196812
       182       45.35   3.50     3  0.077178
       102       44.30   2.50     3  0.056433

#### **Key Take Away**
Unlike **`agg()`**, which usually returns one summary row per group, **`apply()` can return multiple rows per group**.

In [67]:
# Earlier example
result = tips.groupby("smoker")["tip_pct"].describe()
result

,count,mean,std,min,25%,50%,75%,max
smoker,,,,,,,,
No,151.0,0.159328,0.039910,0.056797,0.136906,0.155625,0.185014,0.291990
Yes,93.0,0.163196,0.085119,0.035638,0.106771,0.153846,0.195059,0.710345


In [68]:
# Transpose the summary table
# This changes smoker categories from row labels into column labels
# This is useful here because result is already a DataFrame
result.T

smoker,No,Yes
count,151.000000,93.000000
mean,0.159328,0.163196
std,0.039910,0.085119
min,0.056797,0.035638
25%,0.136906,0.106771
50%,0.155625,0.153846
75%,0.185014,0.195059
max,0.291990,0.710345


#### **Key Take Away**
In this case, `.T` is more appropriate than `unstack()` because `result` is already a DataFrame produced by `describe()`.

Use:

- `.T` when you want to flip rows and columns.
- `unstack()` when you want to move one level of a MultiIndex row label into columns.

### **Filling Missing Values with Group-Specific Values**

Sometimes, replacing missing values with the overall mean is not appropriate.

- For example, if the data belongs to different regions, we may want to fill missing values using the mean of each region instead of the overall mean.

In [69]:
# Create a sample Series with six random values
rng = np.random.default_rng(seed=12345)
s = pd.Series(rng.standard_normal(6))
s

,0
0,-1.423825
1,1.263728
2,-0.870662
3,-0.259173
4,-0.075343
5,-0.740885


In [70]:
# Replace every second value with NaN
s[::2] = np.nan
s

,0
0,NaN
1,1.263728
2,NaN
3,-0.259173
4,NaN
5,-0.740885


In [71]:
# Fill missing values using the overall mean of the Series
s.fillna(s.mean())

,0
0,0.087890
1,1.263728
2,0.087890
3,-0.259173
4,0.087890
5,-0.740885


In [73]:
# Create a mock Series with state names as the index
# Each state is assigned to either the East or West group
states = ["Ohio", "New York", "Vermont", "Florida",
          "Oregon", "Nevada", "California", "Idaho"]
group_key = ["East", "East", "East", "East",
             "West", "West", "West", "West"] # positional alignment
state_values = pd.Series(rng.standard_normal(8), index=states)
state_values

,0
Ohio,-0.466953
New York,-0.060690
Vermont,0.788844
Florida,-1.256668
Oregon,0.575858
Nevada,1.398979
California,1.322298
Idaho,-0.299699


In [74]:
# Insert missing values into selected states
state_values[["Vermont", "Nevada", "Idaho"]] = np.nan
state_values

,0
Ohio,-0.466953
New York,-0.060690
Vermont,NaN
Florida,-1.256668
Oregon,0.575858
Nevada,NaN
California,1.322298
Idaho,NaN


In [76]:
# Count number of rows in each group
state_values.groupby(group_key).size() # inclusive of missing values

,0
East,4
West,4


In [77]:
# Count non-missing values in each group
state_values.groupby(group_key).count()

,0
East,3
West,2


### **Notes: Difference between `size()` and `count()`**
| Function  | What it counts                             | Result meaning          |
| --------- | ------------------------------------------ | ----------------------- |
| `size()`  | Total number of rows in each group         | Includes missing values |
| `count()` | Number of non-missing values in each group | Excludes missing values |

A simple way to comprehend:

- `size()` answers: “How many rows are in this group?”
- `count()` answers: “How many valid, non-missing values are in this group?”

In [78]:
# Define a function to fill missing values using the mean of each group

def fill_mean(group):
    return group.fillna(group.mean())

# Apply the function separately to East and West groups
state_values.groupby(group_key).apply(fill_mean)

East  Ohio         -0.466953
      New York     -0.060690
      Vermont      -0.594770
      Florida      -1.256668
West  Oregon        0.575858
      Nevada        0.949078
      California    1.322298
      Idaho         0.949078
dtype: float64

In [81]:
# Define specific replacement values for each group
fill_values = {
    "East": 0.5,
    "West": -1
}

def fill_func(group):
    return group.fillna(fill_values[group.name]) # fill na value with the values designated in fill_values according to the group.name

In [80]:
# Fill missing values using predefined values for each group
state_values.groupby(group_key).apply(fill_func)

East  Ohio         -0.466953
      New York     -0.060690
      Vermont       0.500000
      Florida      -1.256668
West  Oregon        0.575858
      Nevada       -1.000000
      California    1.322298
      Idaho        -1.000000
dtype: float64

#### **Key Take Away**
**`group.name`** refers to the name of the current group being processed.

For example:

- when pandas processes the `"East"` group, `group.name` is `"East"`
- when pandas processes the `"West"` group, `group.name` is `"West"`

## **14.4 Group Transforms and “Unwrapped” GroupBy Operations**
The **`transform()`** method applies a function to each group, but returns an output with the same length as the original data.

- This is useful when we want to create group-level features while keeping the original row structure.

In [82]:
# An example
df = pd.DataFrame({'key': ['a', 'b', 'c'] * 4,
                   'value': np.arange(12.)})
df

,key,value
0,a,0.0
1,b,1.0
2,c,2.0
3,a,3.0
4,b,4.0
5,c,5.0
6,a,6.0
7,b,7.0
8,c,8.0
9,a,9.0


In [83]:
# Group means by key
g = df.groupby('key')['value']
g.mean() # aggregation function >> get back one value only

,value
key,
a,4.5
b,5.5
c,6.5


In [85]:
# Define a mean function
def get_mean(group):
    return group.mean()

In [86]:
# Apply the custom mean function to each group
# The result has the same number of rows as the original DataFrame
g.transform(get_mean) # return original shape

,value
0,4.5
1,5.5
2,6.5
3,4.5
4,5.5
5,6.5
6,4.5
7,5.5
8,6.5
9,4.5


In [87]:
# Another way - this is faster using default function
g.transform('mean')

,value
0,4.5
1,5.5
2,6.5
3,4.5
4,5.5
5,6.5
6,4.5
7,5.5
8,6.5
9,4.5


#### **Key Take Away**
Although each group has only one mean value, **`transform()`** <font color='red'>**repeats the group mean for every row in that group**.</font>


In [88]:
# Transform can also apply functions that return values with the same length as the input
def times_two(group):
    return group * 2

g.transform(times_two)

,value
0,0.0
1,2.0
2,4.0
3,6.0
4,8.0
5,10.0
6,12.0
7,14.0
8,16.0
9,18.0


### **Group-wise Normalisation**

Group-wise normalisation means standardising values within each group.

- For each value, we subtract the group mean and divide by the group standard deviation.

In [91]:
# Define a function to standardise values within each group
def normalize(x):
    return (x - x.mean()) / x.std()

In [90]:
# Method 1 - faster
g.transform(normalize)

,value
0,-1.161895
1,-1.161895
2,-1.161895
3,-0.387298
4,-0.387298
5,-0.387298
6,0.387298
7,0.387298
8,0.387298
9,1.161895


In [93]:
# Method 2: using apply()
# apply() is more flexible, but transform() is usually clearer for this task
g.apply(normalize)

key    
a    0    -1.161895
     3    -0.387298
     6     0.387298
     9     1.161895
b    1    -1.161895
     4    -0.387298
     7     0.387298
     10    1.161895
c    2    -1.161895
     5    -0.387298
     8     0.387298
     11    1.161895
Name: value, dtype: float64

In [94]:
# A faster way to perform group-wise normalisation
# This avoids using apply() and uses transform() directly
normalized = (
    df['value'] - g.transform('mean')
) / g.transform('std')

normalized

,value
0,-1.161895
1,-1.161895
2,-1.161895
3,-0.387298
4,-0.387298
5,-0.387298
6,0.387298
7,0.387298
8,0.387298
9,1.161895


#### **Key Take Away**
For group-wise normalisation, `transform()` is usually preferred because it returns a result aligned with the original DataFrame rows.

- This makes it easier to add the normalised values back into the original DataFrame.

## **14.5 Comparing `agg()`, `apply()`, and `transform()`**

The three methods are commonly used with `groupby()`, but they serve different purposes.

| Method | Main purpose | Output size | Example use |
|---|---|---|---|
| `agg()` | Summarise each group | Usually smaller than original data | Mean sales by region |
| `apply()` | Apply flexible custom logic | Can be smaller, same size, or larger | Top 3 rows per group |
| `transform()` | Return group-level result for each original row | Same size as original data | Fill each row with its group mean |

### Simple rule

Use:

- `agg()` when you want one summary result per group.
- `apply()` when your operation is more complex.
- `transform()` when you want the result to align with the original rows.

## **14.6 Pivot Tables and Cross-Tabulation**
A **pivot table** is a table used to summarise data across one or more grouping variables.

It usually contains:

| Argument | Meaning |
|---|---|
| `index` | Variables shown as rows |
| `columns` | Variables shown as columns |
| `values` | Numerical values to summarise |
| `aggfunc` | Function used for summarisation |
| `margins` | Adds row and column totals |
| `fill_value` | Replaces missing values in the output |

In [95]:
# From previous tipping dataset
tips.sample(5)

,total_bill,tip,smoker,day,time,size,tip_pct
91,22.49,3.50,No,Fri,Dinner,2,0.155625
50,12.54,2.50,No,Sun,Dinner,2,0.199362
17,16.29,3.71,No,Sun,Dinner,3,0.227747
220,12.16,2.20,Yes,Fri,Lunch,2,0.180921
191,19.81,4.19,Yes,Thur,Lunch,2,0.211509


In [96]:
# Create a pivot table showing the mean size and tip_pct
# Rows: time and day
# Columns: smoker status

tips.pivot_table(
    index=["time", "day"],
    columns="smoker",
    values=["tip_pct", "size"]
)

size             tip_pct          
smoker             No       Yes        No       Yes
time   day                                         
Dinner Fri   2.000000  2.222222  0.139622  0.165347
       Sat   2.555556  2.476190  0.158048  0.147906
       Sun   2.929825  2.578947  0.160113  0.187250
       Thur  2.000000       NaN  0.159744       NaN
Lunch  Fri   3.000000  1.833333  0.187735  0.188937
       Thur  2.500000  2.352941  0.160311  0.163863

In [ ]:
# Use fill_value to display a value when a combination has no observation
# Note: fill_value affects only the displayed pivot table, not the original data

tips.pivot_table(
    index=["time", "size", "smoker"],
    columns="day",
    values="tip_pct",
    fill_value=0
)

day                      Fri       Sat       Sun      Thur
time   size smoker                                        
Dinner 1    No      0.000000  0.137931  0.000000  0.000000
            Yes     0.000000  0.325733  0.000000  0.000000
       2    No      0.139622  0.162705  0.168859  0.159744
            Yes     0.171297  0.148668  0.207893  0.000000
       3    No      0.000000  0.154661  0.152663  0.000000
            Yes     0.000000  0.144995  0.152660  0.000000
       4    No      0.000000  0.150096  0.148143  0.000000
            Yes     0.117750  0.124515  0.193370  0.000000
       5    No      0.000000  0.000000  0.206928  0.000000
            Yes     0.000000  0.106572  0.065660  0.000000
       6    No      0.000000  0.000000  0.103799  0.000000
Lunch  1    No      0.000000  0.000000  0.000000  0.181728
            Yes     0.223776  0.000000  0.000000  0.000000
       2    No      0.000000  0.000000  0.000000  0.166005
            Yes     0.181969  0.000000  0.000000  0.158843
       3    No      0.187735  0.000000  0.000000  0.084246
            Yes     0.000000  0.000000  0.000000  0.204952
       4    No      0.000000  0.000000  0.000000  0.138919
            Yes     0.000000  0.000000  0.000000  0.155410
       5    No      0.000000  0.000000  0.000000  0.121389
       6    No      0.000000  0.000000  0.000000  0.173706

### **Cross-Tabulations: `pd.crosstab()`**

A cross-tabulation is used to count the frequency of combinations between categorical variables.

It is especially useful for answering questions such as:

- How many right-handed and left-handed people are from each country?
- How many smokers and non-smokers appear on each day?

In [97]:
# Load system I/O package
from io import StringIO

In [98]:
# Data consists of strings
handedness_text = """Sample  Nationality  Handedness
1   USA  Right-handed
2   Japan    Left-handed
3   USA  Right-handed
4   Japan    Right-handed
5   Japan    Left-handed
6   Japan    Right-handed
7   USA  Right-handed
8   USA  Left-handed
9   Japan    Right-handed
10  USA  Right-handed"""

In [99]:
# Convert the string data into a pandas DataFrame
# sep=r"\s+" means one or more spaces are used as separators
handedness_df = pd.read_table(StringIO(handedness_text), sep=r"\s+")
handedness_df

,Sample,Nationality,Handedness
0,1,USA,Right-handed
1,2,Japan,Left-handed
2,3,USA,Right-handed
3,4,Japan,Right-handed
4,5,Japan,Left-handed
5,6,Japan,Right-handed
6,7,USA,Right-handed
7,8,USA,Left-handed
8,9,Japan,Right-handed
9,10,USA,Right-handed


In [100]:
# Count handedness frequency by nationality
pd.crosstab(
    handedness_df["Nationality"],
    handedness_df["Handedness"],
    margins=True
)

Handedness,Left-handed,Right-handed,All
Nationality,,,
Japan,2,3,5
USA,1,4,5
All,3,7,10


In [101]:
# Display tips dataset content
tips

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808
...,...,...,...,...,...,...,...
239,29.03,5.92,No,Sat,Dinner,3,0.203927
240,27.18,2.00,Yes,Sat,Dinner,2,0.073584
241,22.67,2.00,Yes,Sat,Dinner,2,0.088222
242,17.82,1.75,No,Sat,Dinner,2,0.098204


In [102]:
# Count smoker and non-smoker observations by time and day
pd.crosstab(
    [tips["time"],
     tips["day"]],
    tips["smoker"],
    margins=True
)

smoker        No  Yes  All
time   day                
Dinner Fri     3    9   12
       Sat    45   42   87
       Sun    57   19   76
       Thur    1    0    1
Lunch  Fri     1    6    7
       Thur   44   17   61
All          151   93  244

#### **Key Take Away**
This table shows how many smoker and non-smoker observations appear for each combination of time and day.

## **Summary**

In this notebook, we learned how to:

1. Use `groupby()` to split data into groups.
2. Apply aggregation functions such as `mean()`, `sum()`, `max()`, and `count()`.
3. Use custom aggregation functions with `agg()`.
4. Use `apply()` for more flexible group operations.
5. Use `transform()` to return group-level results with the same length as the original data.
6. Create pivot tables and cross-tabulations.

### **Key Takeaway**

Use:

- `agg()` when you want summary values.
- `apply()` when you need flexible custom operations.
- `transform()` when you want to keep the same number of rows as the original data.
- `pivot_table()` and `crosstab()` when you want summary tables.

### **That's all for the day :)**